# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook guides you through exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print('Dataset Title:')
print(metadata_obj.name)
print('\nDataset Description:')
print(metadata_obj.description)
print('\nData Collection:', metadata_obj.dataCollection)
print('\nData Limitations:', metadata_obj.dataLimitations)


## 2. Data Overview
Review available record sets, fields, columns and their `@id` values.

In Croissant datasets, each RecordSet and Field has a unique `@id` identifier. This allows us to reference them precisely throughout our analysis.

In [ ]:
# List all RecordSet @ids
record_sets = dataset.record_sets
print('Available record sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']} (name: {rs.get('name', '<no name>')})")

# For each record set, print available fields and columns by @id
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']} (name: {rs.get('name', '<no name>')})")
    fields = rs.get('fields', [])
    if fields:
        print('  Fields:')
        for f in fields:
            print(f"    - @id: {f['@id']} (name: {f.get('name', '<no name>')})")
    columns = rs.get('columns', [])
    if columns:
        print('  Columns:')
        for c in columns:
            print(f"    - @id: {c['@id']} (name: {c.get('name', '<no name>')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record set and field `@id`s.

First, define the list of RecordSet `@id`s, then extract data for each into a pandas DataFrame.

In [ ]:
# Example: list all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print the available columns for each record set
for rid in record_set_ids:
    print(f"\nRecordSet @id: {rid}")
    print('Columns:', dataframes[rid].columns.tolist())
    print(dataframes[rid].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Use field and column `@id`s.

In [ ]:
# Choose a record set for EDA
# Replace <record_set_id> with the desired RecordSet @id from above.
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id is not None:
    df = dataframes[record_set_id]
    print(f"Exploring dataframe for RecordSet @id: {record_set_id}")

    # Select a numeric column for analysis (by @id or column name)
    # Try to guess a numeric column (e.g., 'age at diagnosis', 'height', 'hormone levels', etc.)
    numeric_col_candidates = [col for col in df.columns if ('age' in col.lower() or 'height' in col.lower() or 'level' in col.lower())]

    if numeric_col_candidates:
        numeric_field = numeric_col_candidates[0]
        print(f"Using numeric field: {numeric_field}")

        # Filter records by a threshold
        threshold = 10
        try:
            filtered_df = df[df[numeric_field] > threshold]
        except Exception as e:
            print(f"Error filtering field '{numeric_field}': {e}")
            filtered_df = df.copy()

        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        try:
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        except Exception as e:
            print(f"Error normalizing field '{numeric_field}': {e}")

        # Group by another field
        group_field_candidates = [col for col in df.columns if 'phenotype' in col.lower() or 'diagnosis' in col.lower() or 'infertility' in col.lower()]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No clearly numeric field found for EDA.")
else:
    print("No record set available.")

## 5. Visualization
Visualize numeric field distributions or relationships between fields using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if record_set_id is not None and numeric_col_candidates:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=5, kde=True)
    plt.title(f'Distribution of {numeric_field} in RecordSet @id: {record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Visualize normalized values if available
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=5, kde=True, color='orange')
        plt.title(f'Normalized {numeric_field} Distribution')
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel('Count')
        plt.show()

    # If grouped_df exists, plot group mean
    if 'grouped_df' in locals():
        grouped_df.reset_index(inplace=True)
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.show()


## 6. Conclusion

- The FAIR² dataset provides structured clinical, hormone, imaging, and genetic information for three female patients with non-classical 21-hydroxylase deficiency-related infertility.
- Data was loaded and explored using entity `@id`s as required by Croissant schema best practices.
- EDA and simple visualizations give insights into the distribution and relationships in the clinical/hormone/genotype fields.
- Note the dataset's limitations: very small sample size and single clinical site; generalization is not recommended.